# AppearanceMeetsGeometry — Tier-1 Orchestrator

Drives the full **train → segment → evaluate** pipeline for every model variant,
repeated `N_RUNS` times, to measure run-to-run spread and to carry each of the
five remaining review steps through the same machinery.

**This first run = Step 1**: retrain on the tilt-corrected normal maps and push
all variants through to ROI metrics.

How it works:
- All pipeline logic lives in the `amg_pipeline` package (the single source of
  truth, extracted from the original notebooks which now sit in `Archive/`).
- One `RunConfig` per run; **every output path is derived from a `run_id`**
  (`{channels}ch_run{N}`), so nothing ever overwrites anything.
- Every stage is **skip-if-exists**, so an interrupted sweep resumes cleanly.

Workflow below: (1) set config → (2) verify architecture → (3) preview &
check paths (no training) → (4) run sweep → (5) variance analysis.

> Authoring on macOS, heavy lifting on Windows: the only thing that changes
> between machines is the path block in the CONFIG cell. Training uses CUDA when
> available and falls back to CPU otherwise (matching the original notebooks).

## 0. Imports

In [1]:
import os, sys
import numpy as np
import pandas as pd

# Make the amg_pipeline package importable (it lives one level up, in the repo root).
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

import amg_pipeline as amg
from amg_pipeline.config import RunConfig, make_run_id
from amg_pipeline import paths

print("amg_pipeline loaded from:", os.path.dirname(amg.__file__))

amg_pipeline loaded from: c:\Users\admin\Documents\GitHub\AppearanceMeetsGeometry\amg_pipeline


## 1. CONFIG — set everything here

**This is the only cell you edit between machines / experiments.** Point the
paths at your data, set the experiment name, the variants, the run count, and
the seed. Then verify in the preview cell before running anything heavy.

In [2]:
# === TRAINING CONDITION ===
# "baseline" = frozen Step-1/2 config (no balancing); the control, used for Step 4.
# The other entries are the Step-3 class-imbalance experiments, kept here for reference.
CONDITION = "baseline"   # "baseline" | "oversample" | "dice070" | "both"
SAMPLER_WEIGHT = 3.0       # quarry-tile oversampling weight (only used when sampler is on)
_COND = {
    "baseline":   dict(sampler=False, weight=SAMPLER_WEIGHT, ce=0.5),  # frozen base, Step 4 control
    "oversample": dict(sampler=True,  weight=SAMPLER_WEIGHT, ce=0.5),
    "dice070":    dict(sampler=False, weight=SAMPLER_WEIGHT, ce=0.3),
    "both":       dict(sampler=True,  weight=SAMPLER_WEIGHT, ce=0.3),
}[CONDITION]

# === EXPERIMENT IDENTITY ===
EXPERIMENT_NAME = "v7_photoaug"      # Stage 1 recipe screen: photometric RGB augmentation
EXPERIMENTS_ROOT = os.path.join(REPO_ROOT, "experiments")

# === SWEEP SHAPE ===
CHANNEL_VARIANTS = (4, 7)      # Stage-1 screening pair; 3ch joins in the confirmation sweep
WIDTH = "base"                 # "slim" | "base" | "wide" — Step 3 builds on the base width
N_RUNS = 5                     # runs per variant (change to 10, etc. — no other edits needed)
SEED = 42                      # fixed this round; a later seed sweep is just changing this

# === STAGE-1 RECIPE (one variable at a time) ===
NORM = "none"                  # back to the original architecture (GroupNorm screen: negative)
PHOTO_AUG = "rgb"              # "none" = original training | "rgb" = on-the-fly color jitter (train split only)

# === TRAINING DATA (per the agreed setup) ===
# 3ch uses NORMALMAP_DIR + MASK_DIR; 4ch uses ORTHO_DIR + MASK_DIR; 7ch uses all three.
# 4ch is appearance-only, so its normal-map dir is irrelevant. Point each variant's
# inputs wherever you intend and CHECK in the preview cell before running.
ORTHO_DIR     = r"C:\Users\admin\Desktop\AmG-Artikel2025\PaperVersion2\V2-3_NewNormalmaps_tiltON_epsV1\2025_AmG_TrainingTestingDataCompress\snippets_orthomosaics"
NORMALMAP_DIR = r"C:\Users\admin\Desktop\AmG-Artikel2025\PaperVersion2\V2-3_NewNormalmaps_tiltON_epsV1\2025_AmG_TrainingTestingDataCompress\snippets_normalmaps" 
MASK_DIR      = r"C:\Users\admin\Desktop\AmG-Artikel2025\PaperVersion2\V2-3_NewNormalmaps_tiltON_epsV1\2025_AmG_TrainingTestingDataCompress\snippets_masks"

# === TEST WALLS ===
TEST_ORTHO_DIR     = r"C:\Users\admin\Desktop\AmG-Artikel2025\PaperVersion2\V2-3_NewNormalmaps_tiltON_epsV1\2025_AmG_TrainingTestingDataCompress\Testing\01_test-images"
TEST_NORMALMAP_DIR = r"C:\Users\admin\Desktop\AmG-Artikel2025\PaperVersion2\V2-3_NewNormalmaps_tiltON_epsV1\2025_AmG_TrainingTestingDataCompress\Testing\02_test-normals"
TEST_MASK_DIR      = r"C:\Users\admin\Desktop\AmG-Artikel2025\PaperVersion2\V2-3_NewNormalmaps_tiltON_epsV1\2025_AmG_TrainingTestingDataCompress\Testing\03_test-masks"
WALLS = ["wall1", "wall2", "wall3", "wall4"]

# Filename patterns for each wall ({wall} is substituted). Defaults match the repo.
ORTHO_PATTERN     = "{wall}_png-ortho.png"
NORMALMAP_PATTERN = "{wall}_DEM_normalmap.png"
MASK_PATTERN      = "{wall}_png-ortho.png"   # GT mask reuses the ortho name in this repo

# === HYPER-PARAMS (defaults reproduce the originals) ===
N_EPOCHS   = 300
BATCH_SIZE = 16
LR         = 1e-4

# === ROI EVAL ===
ROI_OPERATION = "closing"
KERNEL_RADIUS = 45

# Build a base config; channels/run_number are filled in per run by the sweep.
BASE_CONFIG = RunConfig(
    channels=7, run_number=1,
    experiment_name=EXPERIMENT_NAME, experiments_root=EXPERIMENTS_ROOT,
    ortho_dir=ORTHO_DIR, normalmap_dir=NORMALMAP_DIR, mask_dir=MASK_DIR,
    test_ortho_dir=TEST_ORTHO_DIR, test_normalmap_dir=TEST_NORMALMAP_DIR,
    test_mask_dir=TEST_MASK_DIR, walls=WALLS,
    ortho_pattern=ORTHO_PATTERN, normalmap_pattern=NORMALMAP_PATTERN, mask_pattern=MASK_PATTERN,
    seed=SEED, n_epochs=N_EPOCHS, batch_size=BATCH_SIZE, lr=LR,
    roi_operation=ROI_OPERATION, kernel_radius=KERNEL_RADIUS,
    width_mult=WIDTH,
    norm=NORM,
    photo_aug=PHOTO_AUG,
    use_weighted_sampler=_COND["sampler"],
    sampler_weight=_COND["weight"],
    ce_weight=_COND["ce"],
)
print("Base config OK. Experiment:", EXPERIMENT_NAME,
      "| norm:", NORM, "| photo_aug:", PHOTO_AUG,
      "| variants:", CHANNEL_VARIANTS, "| runs each:", N_RUNS,
      "| total runs:", len(CHANNEL_VARIANTS) * N_RUNS)

Base config OK. Experiment: v7_photoaug | norm: none | photo_aug: rgb | variants: (4, 7) | runs each: 5 | total runs: 10


In [3]:
# === R3 #7: ensemble confidence pass (Wall 1, 7ch only) ===
import dataclasses
from amg_pipeline.confidence import run_confidence

run_confidence(
    dataclasses.replace(BASE_CONFIG, walls=["wall1"]),
    source_experiment="v2_yaw_correction-epsV1",
    out_experiment="v2_baseline_ens",
    channel_variants=(7,),
    n_runs=5,
    checkpoint_filename="model.pth",   # matches the ens manifest provenance
    probs_wall="wall1",
)

=== confidence v2_baseline_ens 7ch_run1: 5 models from v2_yaw_correction-epsV1 (model.pth) | device=cuda ===


wall1: windows x5 models: 100%|██████████| 85/85 [00:28<00:00,  3.01it/s]


saved c:\Users\admin\Documents\GitHub\AppearanceMeetsGeometry\experiments\v2_baseline_ens\confidence\7ch_run1\wall1_confidence.png
saved c:\Users\admin\Documents\GitHub\AppearanceMeetsGeometry\experiments\v2_baseline_ens\confidence\7ch_run1\wall1_probs.npz
saved c:\Users\admin\Documents\GitHub\AppearanceMeetsGeometry\experiments\v2_baseline_ens\confidence\7ch_run1\confidence_summary.csv


## 2. Verify the extracted model matches the originals

Because the pipeline logic was extracted by hand and the originals are archived,
this proves the model is identical before any sweep runs:
- exact parameter counts + forward-pass shapes for 3/4/7-channel;
- **loads your existing trained checkpoints** and asserts the weights map
  cleanly onto the extracted architecture (the definitive check).

If any of this fails, **stop** — do not run the sweep.

In [3]:
amg.verify_architecture()

# Point these at your existing trained models to prove architecture identity.
# (Adjust filenames if needed; comment out any that aren't present.)
EXISTING_CKPTS = {
    3: os.path.join(REPO_ROOT, "02_MachineLearning", "2025-08-11_3-channel_4-class-EX_300.pth"),
    4: os.path.join(REPO_ROOT, "02_MachineLearning", "2025-08-11_4-channel_4-class-EX_300.pth"),
    7: os.path.join(REPO_ROOT, "02_MachineLearning", "2025-08-11_7-channel_4-class-EX_300.pth"),
}
for ch, p in EXISTING_CKPTS.items():
    if os.path.exists(p):
        amg.verify_checkpoint_loads(p, channels=ch)
    else:
        print(f"(skip) no existing {ch}ch checkpoint at {p}")

[OK ] 3ch base: params=2,158,756 (expected 2,158,756), out=(1, 4, 512, 512)
[OK ] 4ch base: params=2,158,900 (expected 2,158,900), out=(1, 4, 512, 512)
[OK ] 7ch base: params=2,159,332 (expected 2,159,332), out=(1, 4, 512, 512)
Architecture verification PASSED.
(skip) no existing 3ch checkpoint at c:\Users\admin\Documents\GitHub\AppearanceMeetsGeometry\02_MachineLearning\2025-08-11_3-channel_4-class-EX_300.pth
(skip) no existing 4ch checkpoint at c:\Users\admin\Documents\GitHub\AppearanceMeetsGeometry\02_MachineLearning\2025-08-11_4-channel_4-class-EX_300.pth
(skip) no existing 7ch checkpoint at c:\Users\admin\Documents\GitHub\AppearanceMeetsGeometry\02_MachineLearning\2025-08-11_7-channel_4-class-EX_300.pth


## 3. Preview & check paths — NO training

Inspect-before-write gate. This derives every output path, confirms the training
directories exist and how many tiles they hold, and checks that every test-wall
input file is present. **Fix any ✗ here before running the sweep.**

In [4]:
import glob, dataclasses

def _count_pngs(d):
    return len(glob.glob(os.path.join(d, "*.png"))) if os.path.isdir(d) else None

print("=== TRAINING DIRECTORIES ===")
for label, d in [("ortho", ORTHO_DIR), ("normalmap", NORMALMAP_DIR), ("mask", MASK_DIR)]:
    n = _count_pngs(d)
    print(f"  {'OK ' if n else '✗  '} {label:10s} {d}  ->  {n if n is not None else 'MISSING'} png")

print("\n=== TEST WALL INPUTS ===")
all_ok = True
for wall in WALLS:
    for label, fn in [("ortho", paths.test_ortho_path(BASE_CONFIG, wall)),
                      ("normal", paths.test_normalmap_path(BASE_CONFIG, wall)),
                      ("mask", paths.test_mask_path(BASE_CONFIG, wall))]:
        exists = os.path.exists(fn)
        all_ok = all_ok and exists
        print(f"  {'OK ' if exists else '✗  '} {wall} {label:7s} {fn}")

print("\n=== DERIVED OUTPUT PATHS (sample: 7ch_run1) ===")
sample = dataclasses.replace(BASE_CONFIG, channels=7, run_number=1)
print("  checkpoint :", paths.checkpoint_path(sample))
print("  config.json:", paths.config_json_path(sample))
print("  seg (wall1):", paths.segmentation_raw_path(sample, "wall1"))
print("  metrics dir:", paths.metrics_wall_dir(sample, "wall1"))
print("  manifest   :", paths.manifest_path(sample))

print("\n=== RUN IDS TO BE PRODUCED ===")
for cfg in amg.build_configs(BASE_CONFIG, CHANNEL_VARIANTS, N_RUNS):
    print("   ", make_run_id(cfg))

print("\nAll test inputs present." if all_ok else "\n*** Some inputs MISSING — fix before running. ***")

=== TRAINING DIRECTORIES ===
  OK  ortho      C:\Users\admin\Desktop\AmG-Artikel2025\PaperVersion2\V2-3_NewNormalmaps_tiltON_epsV1\2025_AmG_TrainingTestingDataCompress\snippets_orthomosaics  ->  2149 png
  OK  normalmap  C:\Users\admin\Desktop\AmG-Artikel2025\PaperVersion2\V2-3_NewNormalmaps_tiltON_epsV1\2025_AmG_TrainingTestingDataCompress\snippets_normalmaps  ->  2149 png
  OK  mask       C:\Users\admin\Desktop\AmG-Artikel2025\PaperVersion2\V2-3_NewNormalmaps_tiltON_epsV1\2025_AmG_TrainingTestingDataCompress\snippets_masks  ->  2149 png

=== TEST WALL INPUTS ===
  OK  wall1 ortho   C:\Users\admin\Desktop\AmG-Artikel2025\PaperVersion2\V2-3_NewNormalmaps_tiltON_epsV1\2025_AmG_TrainingTestingDataCompress\Testing\01_test-images\wall1_png-ortho.png
  OK  wall1 normal  C:\Users\admin\Desktop\AmG-Artikel2025\PaperVersion2\V2-3_NewNormalmaps_tiltON_epsV1\2025_AmG_TrainingTestingDataCompress\Testing\02_test-normals\wall1_DEM_normalmap.png
  OK  wall1 mask    C:\Users\admin\Desktop\AmG-Artikel

## 4. Run the sweep

Gated by `DO_RUN` so it can't fire by accident. Each run does
train → segment(all walls) → evaluate(all walls) → append manifest, and every
stage skips work already on disk, so re-running resumes an interrupted sweep.

Stage gates (`DO_TRAIN/DO_SEGMENT/DO_EVALUATE`) let you run stages separately —
e.g. recompute metrics without retraining by setting `DO_TRAIN=DO_SEGMENT=False`.

In [ ]:
import dataclasses

DO_RUN = True        # <-- already armed: Run All launches the sweep
DO_TRAIN = True
DO_SEGMENT = True
DO_EVALUATE = True
FORCE = False         # True ignores skip-if-exists and recomputes everything

# Stage-1 recipe screen (currently: v7_photoaug): ONE experiment, the screened
# variable set in the CONFIG cell, (4,7)ch x N_RUNS runs. Compare afterwards
# against the frozen v2_yaw_correction-epsV1 manifest. Resume-safe: re-running
# skips finished work.
if DO_RUN:
    amg.run_sweep(BASE_CONFIG, channel_variants=CHANNEL_VARIANTS, n_runs=N_RUNS,
                  do_train=DO_TRAIN, do_segment=DO_SEGMENT, do_evaluate=DO_EVALUATE,
                  force=FORCE)
    print("=== SCREEN COMPLETE ===")
else:
    total = len(CHANNEL_VARIANTS) * N_RUNS
    print(f"DO_RUN is False; set it to True to launch {total} runs "
          f"({len(CHANNEL_VARIANTS)} variants x {N_RUNS} runs).")


## 5. Variance analysis

Loads the master manifest and reports mean ± std (and min/max) of IoU/F1 across
the `N_RUNS` repeats, per variant per wall. This is the spread the whole exercise
is about — it tells you whether a single-run comparison between variants is
trustworthy or within the noise band. Remember: with the seed fixed, this is
run-time/implementation variance (init + split held fixed).

In [6]:
mf_path = paths.manifest_path(BASE_CONFIG)
if not os.path.exists(mf_path):
    print("No manifest yet — run the sweep first:", mf_path)
else:
    mf = pd.read_csv(mf_path)
    metrics = ["IoU_mean_stones", "IoU_Ashlar", "IoU_Polygonal", "IoU_Quarry", "macro_F1"]
    agg = (mf.groupby(["channels", "wall"])[metrics]
             .agg(["mean", "std", "min", "max"]))
    pd.set_option("display.width", 200, "display.max_columns", 50)

    print("=== SPREAD ACROSS RUNS (AllWalls aggregate) ===")
    allw = agg.xs("AllWalls", level="wall")
    for ch in sorted(mf["channels"].unique()):
        row = allw.loc[ch]
        m, s = row[("IoU_mean_stones", "mean")], row[("IoU_mean_stones", "std")]
        lo, hi = row[("IoU_mean_stones", "min")], row[("IoU_mean_stones", "max")]
        print(f"  {ch}ch  mean-stone IoU = {m:.4f} ± {s:.4f}  (min {lo:.4f}, max {hi:.4f})")

    print("\n=== FULL TABLE (mean ± std per variant per wall) ===")
    display(agg.round(4))

    n_per = mf[mf.wall == "AllWalls"].groupby("channels").size()
    print("\nRuns completed per variant:\n", n_per.to_string())

=== SPREAD ACROSS RUNS (AllWalls aggregate) ===
  4ch  mean-stone IoU = 0.5368 ± 0.0057  (min 0.5299, max 0.5447)
  7ch  mean-stone IoU = 0.5602 ± 0.0103  (min 0.5453, max 0.5695)

=== FULL TABLE (mean ± std per variant per wall) ===


IoU_mean_stones                         IoU_Ashlar                         IoU_Polygonal                         IoU_Quarry                         macro_F1                        
                             mean     std     min     max       mean     std     min     max          mean     std     min     max       mean     std     min     max     mean     std     min     max
channels wall                                                                                                                                                                                         
4        AllWalls          0.5368  0.0057  0.5299  0.5447     0.9279  0.0056  0.9194  0.9327        0.5135  0.0093  0.5037  0.5238     0.1690  0.0126  0.1609  0.1909   0.5794  0.0060  0.5715  0.5883
         wall1             0.6284  0.0178  0.6119  0.6553     0.8845  0.0173  0.8583  0.9034        0.6532  0.0164  0.6369  0.6744     0.3475  0.0628  0.3065  0.4567   0.7473  0.0198  0.7314  0.7798
         wall2             0.4973  0.0125  0.4769  0.5112     0.9528  0.0030  0.9491  0.9566           NaN     NaN     NaN     NaN     0.0418  0.0248  0.0000  0.0658   0.5276  0.0234  0.4882  0.5506
         wall3             0.5188  0.0209  0.4931  0.5429        NaN     NaN     NaN     NaN        0.8045  0.0232  0.7697  0.8278     0.2331  0.0299  0.2049  0.2681   0.6344  0.0222  0.6129  0.6595
         wall4             0.3610  0.0092  0.3477  0.3681     0.9464  0.0017  0.9443  0.9488        0.0827  0.0239  0.0448  0.1006     0.0537  0.0317  0.0000  0.0809   0.4084  0.0170  0.3832  0.4210
7        AllWalls          0.5602  0.0103  0.5453  0.5695     0.9233  0.0096  0.9137  0.9359        0.5414  0.0180  0.5198  0.5605     0.2158  0.0157  0.2016  0.2376   0.6005  0.0110  0.5871  0.6160
         wall1             0.6882  0.0169  0.6687  0.7087     0.8843  0.0145  0.8682  0.9060        0.6567  0.0495  0.5828  0.7163     0.5237  0.0326  0.4795  0.5552   0.8058  0.0116  0.7933  0.8184
         wall2             0.4816  0.0069  0.4748  0.4932     0.9534  0.0074  0.9445  0.9627           NaN     NaN     NaN     NaN     0.0098  0.0182  0.0000  0.0418   0.4975  0.0161  0.4871  0.5258
         wall3             0.5645  0.0321  0.5275  0.6103        NaN     NaN     NaN     NaN        0.8530  0.0177  0.8266  0.8697     0.2761  0.0502  0.2284  0.3549   0.6757  0.0341  0.6385  0.7260
         wall4             0.3668  0.0173  0.3419  0.3841     0.9322  0.0087  0.9221  0.9413        0.1144  0.0175  0.0918  0.1352     0.0538  0.0475  0.0086  0.1159   0.4230  0.0290  0.3827  0.4528


Runs completed per variant:
 channels
4    5
7    5
